# Physics Explosion Analysis
Tracks per-run explosion rates and breakdown by cause (pre/post-step NaN/OOB/velocity checks).

Sections:
1. Load runs
2. Overall explosion rate per run
3. Explosion rate vs success rate scatter
4. Explosion cause breakdown
5. Explosion rate over training (time series)
6. Correlation of explosion rate with reward/env config

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
from pathlib import Path

LOGS_DIR = Path('../../logs')

# Optional: restrict to a subset of run IDs (None = load all)
RUN_FILTER: list[str] | None = None

# Only show runs with mean explosion rate above this threshold in the time-series plot
TIMESERIES_MIN_EXPLOSION_RATE = 0.001

In [ ]:
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import yaml

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.5f}'.format)


def strip_omegaconf(text: str) -> str:
    pattern = re.compile(r'\$\{[^{}]+\}')
    for _ in range(10):
        prev, text = text, pattern.sub('null', text)
        if text == prev:
            break
    return text


def load_config(run_dir: Path) -> dict:
    p = run_dir / 'config.yaml'
    if not p.exists():
        return {}
    try:
        return yaml.safe_load(strip_omegaconf(p.read_text())) or {}
    except Exception:
        return {}


def load_csv_safe(path: Path) -> pd.DataFrame | None:
    try:
        df = pd.read_csv(path)
        return df if not df.empty else None
    except Exception:
        return None


EXPLOSION_COLS = [
    'explosion/pct_exploded',
    'explosion/pre_pos_nan',
    'explosion/pre_lin_vel_nan',
    'explosion/pre_pos_out',
    'explosion/pre_lin_vel_high',
    'explosion/pre_ang_vel_high',
    'explosion/pre_obs_nan',
    'explosion/post_pos_nan',
    'explosion/post_pos_out',
    'explosion/post_lin_vel_nan',
    'explosion/post_lin_vel_high',
    'explosion/post_ang_vel_nan',
    'explosion/post_ang_vel_high',
    'explosion/post_orient_nan',
]
CAUSE_COLS = [c for c in EXPLOSION_COLS if c != 'explosion/pct_exploded']
SHORT = {c: c.split('/')[-1] for c in EXPLOSION_COLS}


def collect_runs(logs_dir: Path, run_filter: list[str] | None) -> tuple[pd.DataFrame, dict]:
    """Returns (summary_df, {run_id: explosion_timeseries_df})."""
    records, timeseries = [], {}
    skip = ('eval_', 'host_', 'isaac_', 'wandb')

    def _process(run_dir: Path, run_type: str):
        if run_filter and run_dir.name not in run_filter:
            return
        exp_df = load_csv_safe(run_dir / 'explosion.csv')
        if exp_df is None:
            return
        eval_df  = load_csv_safe(run_dir / 'eval.csv')
        cfg      = load_config(run_dir)
        env_cfg  = cfg.get('env_cfg_overrides', {})

        avail = [c for c in EXPLOSION_COLS if c in exp_df.columns]
        ts = exp_df[['collected_frames'] + avail].copy()
        ts['run_id'] = run_dir.name
        timeseries[run_dir.name] = ts

        row = {
            'run_id':        run_dir.name,
            'run_type':      run_type,
            'name':          cfg.get('name', ''),
            'terrain':       cfg.get('terrain', ''),
            'num_robots':    cfg.get('num_robots'),
            'total_frames':  exp_df['collected_frames'].max() if 'collected_frames' in exp_df.columns else None,
            'n_batches':     len(exp_df),
            'best_success':  eval_df['eval/success_rate'].max() if eval_df is not None and 'eval/success_rate' in eval_df.columns else float('nan'),
        }
        for c in avail:
            row[SHORT[c]] = exp_df[c].mean()
        # env config columns of interest
        for key in ('shock_coef', 'shock_scale', 'roll_coef', 'pitch_coef',
                    'step_penalty', 'shaping_coef', 'goal_reached_reward',
                    'max_depenetration_velocity'):
            row[f'cfg/{key}'] = env_cfg.get(key, cfg.get(key))
        records.append(row)

    for entry in sorted(logs_dir.iterdir()):
        if not entry.is_dir() or any(entry.name.startswith(p) for p in skip):
            continue
        if entry.name.startswith('optuna_'):
            for trial in sorted(entry.iterdir()):
                if trial.is_dir() and re.match(r'.+_\d+$', trial.name):
                    _process(trial, 'optuna')
        elif entry.name.startswith('train_'):
            _process(entry, 'train')

    summary = pd.DataFrame(records).sort_values('pct_exploded', ascending=False).reset_index(drop=True)
    return summary, timeseries


summary, timeseries = collect_runs(LOGS_DIR, RUN_FILTER)
print(f"Loaded {len(summary)} runs with explosion logs")
print(f"Runs with any explosions: {(summary['pct_exploded'] > 0).sum()}")
print(f"Mean pct_exploded across all runs: {summary['pct_exploded'].mean():.5f}")

## Overall explosion rate per run

In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────────
cause_cols_short = [SHORT[c] for c in CAUSE_COLS if SHORT[c] in summary.columns]
display_cols = ['run_id', 'run_type', 'name', 'terrain', 'total_frames',
                'best_success', 'pct_exploded'] + cause_cols_short
display_cols = [c for c in display_cols if c in summary.columns]

with pd.option_context('display.max_rows', None, 'display.width', 250):
    display(summary[display_cols].head(40))

In [ ]:
# ── Bar chart — top 30 worst runs by mean explosion rate ──────────────────────
top30 = summary.nlargest(30, 'pct_exploded')
if top30['pct_exploded'].max() == 0:
    print("No explosions detected in any run.")
else:
    fig, ax = plt.subplots(figsize=(14, 5))
    colors = cm.Reds(np.linspace(0.4, 0.9, len(top30)))[::-1]
    ax.bar(range(len(top30)), top30['pct_exploded'].values, color=colors, edgecolor='black', linewidth=0.4)
    ax.set_xticks(range(len(top30)))
    ax.set_xticklabels(top30['run_id'].values, rotation=60, ha='right', fontsize=7)
    ax.set_ylabel('Mean pct_exploded (fraction of robots per batch)', fontsize=10)
    ax.set_title('Top 30 runs by mean explosion rate', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    fig.tight_layout()
    plt.show()

## Explosion rate vs success rate

In [ ]:
sub = summary.dropna(subset=['pct_exploded', 'best_success'])
if len(sub) == 0:
    print("No runs with both explosion and eval data.")
else:
    fig, ax = plt.subplots(figsize=(9, 6))
    sc = ax.scatter(
        sub['pct_exploded'], sub['best_success'],
        c=sub['pct_exploded'], cmap='Reds', s=60,
        edgecolors='black', linewidths=0.4, alpha=0.85, zorder=3
    )
    fig.colorbar(sc, ax=ax, label='mean pct_exploded')
    ax.set_xlabel('Mean explosion rate (fraction of robots per batch)', fontsize=11)
    ax.set_ylabel('Best success rate', fontsize=11)
    ax.set_title('Explosion Rate vs Success Rate', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

    if sub['pct_exploded'].std() > 0:
        from numpy.polynomial import polynomial as P
        x, y = sub['pct_exploded'].values, sub['best_success'].values
        coef = P.polyfit(x, y, 1)
        xfit = np.linspace(x.min(), x.max(), 200)
        ax.plot(xfit, P.polyval(xfit, coef), '--', color='black', linewidth=1.5,
                label=f'linear fit  r={np.corrcoef(x, y)[0,1]:.2f}')
        ax.legend(fontsize=9)

    fig.tight_layout()
    plt.show()

## Explosion cause breakdown

In [ ]:
# ── Stacked bar: mean rate per cause across all runs ──────────────────────────
cause_short = [SHORT[c] for c in CAUSE_COLS if SHORT[c] in summary.columns]
cause_means = summary[cause_short].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: mean rates
ax = axes[0]
ax.bar(range(len(cause_means)), cause_means.values,
       color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(cause_means)))
ax.set_xticklabels(cause_means.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Mean rate (fraction of robots)', fontsize=10)
ax.set_title('Mean explosion cause rates (all runs)', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Right: only runs with pct_exploded > 0
ax = axes[1]
exploded = summary[summary['pct_exploded'] > 0]
if len(exploded) == 0:
    ax.text(0.5, 0.5, 'No explosions', ha='center', va='center', transform=ax.transAxes, fontsize=12)
else:
    cause_means_exp = exploded[cause_short].mean().sort_values(ascending=False)
    ax.bar(range(len(cause_means_exp)), cause_means_exp.values,
           color='tomato', edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(cause_means_exp)))
    ax.set_xticklabels(cause_means_exp.index, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Mean rate (fraction of robots)', fontsize=10)
    ax.set_title(f'Mean explosion cause rates (runs with explosions, n={len(exploded)})',
                 fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

fig.tight_layout()
plt.show()

In [ ]:
# ── Heatmap: cause rates for top-exploding runs ────────────────────────────────
top_exp = summary[summary['pct_exploded'] > 0].nlargest(30, 'pct_exploded')
if len(top_exp) == 0:
    print("No runs with explosions to display.")
else:
    avail_causes = [c for c in cause_short if c in top_exp.columns]
    heat_data = top_exp.set_index('run_id')[avail_causes]

    fig, ax = plt.subplots(figsize=(max(10, len(avail_causes) * 0.8), max(5, len(top_exp) * 0.4)))
    im = ax.imshow(heat_data.values, cmap='Reds', aspect='auto',
                   vmin=0, vmax=heat_data.values.max())
    ax.set_xticks(range(len(avail_causes)))
    ax.set_xticklabels(avail_causes, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(heat_data)))
    ax.set_yticklabels(heat_data.index, fontsize=7)
    for i in range(len(heat_data)):
        for j in range(len(avail_causes)):
            val = heat_data.values[i, j]
            ax.text(j, i, f'{val:.4f}', ha='center', va='center',
                    fontsize=6, color='white' if val > heat_data.values.max() * 0.6 else 'black')
    fig.colorbar(im, ax=ax, shrink=0.6, label='Mean rate')
    ax.set_title('Explosion causes — top-exploding runs (sorted by pct_exploded)',
                 fontsize=11, fontweight='bold')
    fig.tight_layout()
    plt.show()

## Explosion rate over training

In [ ]:
# ── Time series for runs with notable explosions ───────────────────────────────
notable = summary[summary['pct_exploded'] >= TIMESERIES_MIN_EXPLOSION_RATE]['run_id'].tolist()
print(f"{len(notable)} runs with mean pct_exploded >= {TIMESERIES_MIN_EXPLOSION_RATE}")

if not notable:
    print("No notable explosions — lower TIMESERIES_MIN_EXPLOSION_RATE to see more runs.")
else:
    fig, ax = plt.subplots(figsize=(14, 6))
    cmap = cm.get_cmap('tab20', len(notable))
    for i, run_id in enumerate(notable):
        ts = timeseries[run_id]
        if 'explosion/pct_exploded' not in ts.columns:
            continue
        ax.plot(ts['collected_frames'], ts['explosion/pct_exploded'],
                linewidth=1.2, alpha=0.8, color=cmap(i), label=run_id)

    ax.set_xlabel('Collected frames', fontsize=11)
    ax.set_ylabel('pct_exploded', fontsize=11)
    ax.set_title('Explosion rate over training', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    if len(notable) <= 15:
        ax.legend(fontsize=7, loc='upper right', ncol=2)
    fig.tight_layout()
    plt.show()

In [ ]:
# ── Cause time series for single worst run ────────────────────────────────────
if notable:
    worst_run = summary.loc[summary['run_id'].isin(notable), 'pct_exploded'].idxmax()
    worst_id  = summary.loc[worst_run, 'run_id']
    ts = timeseries[worst_id]
    avail_ts_causes = [c for c in CAUSE_COLS if c in ts.columns]

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # Total explosion rate
    if 'explosion/pct_exploded' in ts.columns:
        axes[0].plot(ts['collected_frames'], ts['explosion/pct_exploded'],
                     linewidth=2, color='black', label='pct_exploded')
        axes[0].set_ylabel('pct_exploded', fontsize=10)
        axes[0].set_title(f'{worst_id} — explosion rate + cause breakdown', fontsize=11, fontweight='bold')
        axes[0].grid(True, alpha=0.3)
        axes[0].legend(fontsize=9)

    # Cause breakdown
    ax = axes[1]
    pre_causes  = [c for c in avail_ts_causes if 'pre_'  in c]
    post_causes = [c for c in avail_ts_causes if 'post_' in c]
    for c in pre_causes:
        ax.plot(ts['collected_frames'], ts[c], linestyle='-',  linewidth=1.2, alpha=0.8, label=SHORT[c])
    for c in post_causes:
        ax.plot(ts['collected_frames'], ts[c], linestyle='--', linewidth=1.0, alpha=0.7, label=SHORT[c])
    ax.set_xlabel('Collected frames', fontsize=10)
    ax.set_ylabel('Cause rate', fontsize=10)
    ax.set_title('Explosion causes (solid=pre, dashed=post)', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, ncol=3, loc='upper right')

    fig.tight_layout()
    plt.show()
    print(f"Shown: {worst_id}")

## Correlation with reward / env config

In [ ]:
# ── Correlation of pct_exploded with config knobs ─────────────────────────────
cfg_cols = [c for c in summary.columns if c.startswith('cfg/')]
num_cols = cfg_cols + ['pct_exploded', 'best_success']
num_cols = [c for c in num_cols if summary[c].dtype != object and summary[c].notna().sum() >= 5]

if len(num_cols) < 2:
    print("Not enough numeric columns for correlation analysis.")
else:
    corr = summary[num_cols].corr()
    fig, ax = plt.subplots(figsize=(max(8, len(num_cols) * 0.65), max(6, len(num_cols) * 0.6)))
    im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    labels = [c.replace('cfg/', '') for c in num_cols]
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    for i in range(len(labels)):
        for j in range(len(labels)):
            val = corr.values[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=6, color='white' if abs(val) > 0.6 else 'black')
    fig.colorbar(im, ax=ax, shrink=0.8, label='Pearson r')
    ax.set_title('Correlation: explosion rate + success vs config', fontsize=11, fontweight='bold')
    fig.tight_layout()
    plt.show()

    print("\nTop correlations with pct_exploded:")
    exp_corr = corr['pct_exploded'].drop('pct_exploded').abs().sort_values(ascending=False)
    for name, val in exp_corr.items():
        r = corr.loc['pct_exploded', name]
        print(f"  {name.replace('cfg/', ''):40s}  r = {r:+.3f}")